# K-mer Feature Pipeline (Counts)

This notebook repeats the selection & modeling flow but uses actual k-mer counts
for the final modeling step (with `log1p` scaling). Selection still uses binary
presence/absence for robustness.

In [ ]:
from __future__ import annotations

import json
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_selection import chi2
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from sklearn.preprocessing import StandardScaler
import joblib

## Helpers
Reuse the same `iter_genome_kmers`, `build_prevalence`, and selection helpers as the
binary notebook. The final matrix will be built with counts and transformed.

In [ ]:
# Helper utilities for k-mer pipelines (binary-presence notebook)
# Each function below includes a short comment/docstring explaining its role, inputs, and outputs.
from typing import Iterable

def iter_genome_kmers(dump_path: Path) -> dict[str, int]:
    """Read a per-genome k-mer dump and return a dict of {kmer: count}.

    Parameters:
    - dump_path: Path to a single genome's k-mer dump file (two columns: kmer count).

    Returns:
    - dict mapping kmer (str) to integer count.
    """
    kmers: dict[str, int] = {}
    with dump_path.open("r", encoding="utf8", errors="ignore") as fh:
        for line in fh:
            parts = line.strip().split()
            if len(parts) != 2:
                continue
            kmer, cnt = parts
            try:
                kmers[kmer] = int(cnt)
            except ValueError:
                continue
    return kmers


def build_prevalence(dump_dir: str, genome_ids: list[str]) -> Counter:
    """Aggregate presence counts for each k-mer across a list of genomes.

    Scans each genome's k-mer dump and counts how many genomes contain each k-mer (presence, not counts).

    Parameters:
    - dump_dir: directory containing `{GenomeID}_db_kmers.txt` files.
    - genome_ids: list of genome identifiers to include.

    Returns:
    - Counter where keys are k-mers and values are the number of genomes containing that k-mer.
    """
    prevalence: Counter = Counter()
    dump_dir = Path(dump_dir)
    for gid in genome_ids:
        dump_path = dump_dir / f"{gid}_db_kmers.txt"
        if not dump_path.exists():
            continue
        kmers = iter_genome_kmers(dump_path)
        prevalence.update(kmers.keys())
    return prevalence


def select_vocab_by_prevalence(prevalence: Counter, n_genomes: int,
                               min_frac: float = 0.02, max_frac: float = 0.95) -> list[str]:
    """Filter k-mers by prevalence across genomes to remove extremely rare or ubiquitous features.

    Parameters:
    - prevalence: Counter from `build_prevalence` mapping k-mer -> genome occurrence count.
    - n_genomes: number of genomes used to compute prevalence (for fraction -> absolute conversion).
    - min_frac: minimum fraction of genomes that must contain a k-mer to keep it.
    - max_frac: maximum fraction of genomes that may contain a k-mer to keep it.

    Returns:
    - List of k-mer strings that pass the prevalence filters.
    """
    min_count = int(np.ceil(min_frac * n_genomes))
    max_count = int(np.floor(max_frac * n_genomes))
    vocab = [k for k, c in prevalence.items() if min_count <= c <= max_count]
    return vocab


def build_sparse_matrix(
    dump_dir: str,
    genome_ids: list[str],
    vocab: list[str],
    binary: bool = True,
    chunk_size: int = 100,
) -> sparse.csr_matrix:
    """Construct a sparse (CSR) matrix of shape (n_genomes, n_features) from k-mer dumps.

    This reads each genome's dump and populates the matrix using the provided `vocab` index.
    When `binary` is True, presence is recorded as 1; otherwise counts are used (int).
    The build is chunked to keep peak memory low.

    Parameters:
    - dump_dir: directory with per-genome k-mer dump files.
    - genome_ids: ordered list of genome IDs corresponding to rows in the matrix.
    - vocab: ordered list of k-mers corresponding to columns in the matrix.
    - binary: whether to collapse counts to binary presence/absence.
    - chunk_size: number of genomes per chunk when building the matrix.

    Returns:
    - scipy.sparse.csr_matrix with dtype `np.int8` for binary (or `np.int32` for counts).
    """
    vocab_index = {kmer: idx for idx, kmer in enumerate(vocab)}
    dump_dir = Path(dump_dir)
    blocks = []
    dtype = np.int8 if binary else np.int32
    n_features = len(vocab)

    for start in range(0, len(genome_ids), chunk_size):
        chunk_ids = genome_ids[start:start + chunk_size]
        rows: list[int] = []
        cols: list[int] = []
        data: list[int] = []
        for row_idx, gid in enumerate(chunk_ids):
            dump_path = dump_dir / f"{gid}_db_kmers.txt"
            if not dump_path.exists():
                continue
            kmers = iter_genome_kmers(dump_path)
            for kmer, cnt in kmers.keys():
                col_idx = vocab_index.get(kmer)
                if col_idx is None:
                    continue
                rows.append(row_idx)
                cols.append(col_idx)
                data.append(1 if binary else int(cnt))
        if rows:
            block = sparse.csr_matrix((data, (rows, cols)), shape=(len(chunk_ids), n_features), dtype=dtype)
        else:
            block = sparse.csr_matrix((len(chunk_ids), n_features), dtype=dtype)
        blocks.append(block)
    if not blocks:
        return sparse.csr_matrix((0, n_features), dtype=dtype)
    return sparse.vstack(blocks, format='csr')


def load_labels(labels_path: Path) -> pd.Series:
    """Load phenotype labels from a CSV and return a binary series indexed by GenomeID.

    Expects a CSV with at least `Genome ID` and `phenotype` columns. Optionally maps 'I' -> 'R'.

    Parameters:
    - labels_path: Path to CSV containing labels.

    Returns:
    - pandas Series indexed by GenomeID with values 1 for resistant and 0 for susceptible.
    """
    df = pd.read_csv(labels_path)

    # Normalize column lookup so small header variations do not break the notebook.
    norm = {c.strip().lower(): c for c in df.columns}
    gid_col = norm.get("genome id")
    pheno_col = norm.get("phenotype")

    if gid_col is None or pheno_col is None:
        raise ValueError(
            "Expected columns for Genome ID and phenotype in labels file. "
            f"Found columns: {list(df.columns)}"
        )

    pheno = df.set_index(gid_col)[pheno_col]
    pheno = pd.to_numeric(pheno, errors="coerce")
    return pheno

In [ ]:
# # Helper utilities for k-mer pipelines (counts notebook)
# # Each function includes a short docstring explaining purpose, parameters, and return value.
# def iter_genome_kmers(dump_path: Path) -> dict[str, int]:
#     """Read a per-genome k-mer dump and return a dict of {kmer: count}.

#     Parameters:
#     - dump_path: Path to a single genome's k-mer dump file (two columns: kmer count).

#     Returns:
#     - dict mapping kmer (str) to integer count.
#     """
#     kmers: dict[str, int] = {}
#     with dump_path.open('r', encoding='utf8', errors='ignore') as f:
#         for line in f:
#             parts = line.strip().split()
#             if len(parts) != 2:
#                 continue
#             kmer, cnt = parts
#             try:
#                 kmers[kmer] = int(cnt)
#             except ValueError:
#                 continue
#     return kmers

# def build_prevalence(dump_dir: str, genome_ids: list[str]) -> Counter:
#     """Aggregate presence counts for each k-mer across a list of genomes.

#     Scans each genome's k-mer dump and counts how many genomes contain each k-mer (presence, not counts).

#     Parameters:
#     - dump_dir: directory containing `{GenomeID}_db_kmers.txt` files.
#     - genome_ids: list of genome identifiers to include.

#     Returns:
#     - Counter where keys are k-mers and values are the number of genomes containing that k-mer.
#     """
#     prevalence = Counter()
#     dump_dir = Path(dump_dir)
#     for gid in genome_ids:
#         dump_path = dump_dir / f'{gid}_db_kmers.txt'
#         if not dump_path.exists():
#             continue
#         kmers = iter_genome_kmers(dump_path)
#         # update counts by presence (keys only)
#         prevalence.update(kmers.keys())
#     return prevalence

# def select_vocab_by_prevalence(prevalence: Counter, n_genomes: int,
#                                    min_frac: float = 0.01, max_frac: float = 0.99) -> list[str]:
#     """Filter k-mers by prevalence across genomes to remove extremely rare or ubiquitous features.

#     Parameters:
#     - prevalence: Counter from `build_prevalence` mapping k-mer -> genome occurrence count.
#     - n_genomes: number of genomes used to compute prevalence (for fraction -> absolute conversion).
#     - min_frac: minimum fraction of genomes that must contain a k-mer to keep it.
#     - max_frac: maximum fraction of genomes that may contain a k-mer to keep it.

#     Returns:
#     - List of k-mer strings that pass the prevalence filters.
#     """
#     min_count = int(np.ceil(min_frac * n_genomes))
#     max_count = int(np.floor(max_frac * n_genomes))
#     vocab = [k for k, c in prevalence.items() if min_count <= c <= max_count]
#     return vocab

# def build_sparse_matrix(dump_dir: str, genome_ids: list[str], vocab: list[str], binary: bool = False) -> sparse.csr_matrix:
#     """Construct a sparse (CSR) matrix of shape (n_genomes, n_features) from k-mer dumps.

#     Reads each genome's dump and populates the matrix using the provided `vocab` index.
#     When `binary` is True, presence is recorded as 1; otherwise counts are used (int).

#     Parameters:
#     - dump_dir: directory with per-genome k-mer dump files.
#     - genome_ids: ordered list of genome IDs corresponding to rows in the matrix.
#     - vocab: ordered list of k-mers corresponding to columns in the matrix.
#     - binary: whether to collapse counts to binary presence/absence.

#     Returns:
#     - scipy.sparse.csr_matrix with dtype `np.int32` for counts (or `np.int8` for binary).
#     """
#     vocab_index = {kmer: idx for idx, kmer in enumerate(vocab)}
#     rows = []
#     cols = []
#     data = []
#     dump_dir = Path(dump_dir)
#     for row_idx, gid in enumerate(genome_ids):
#         dump_path = dump_dir / f'{gid}_db_kmers.txt'
#         if not dump_path.exists():
#             continue
#         kmers = iter_genome_kmers(dump_path)
#         for kmer, cnt in kmers.items():
#             col_idx = vocab_index.get(kmer)
#             if col_idx is None:
#                 continue
#             rows.append(row_idx)
#             cols.append(col_idx)
#             data.append(1 if binary else int(cnt))
#     dtype = np.int8 if binary else np.int32
#     mat = sparse.csr_matrix((data, (rows, cols)), shape=(len(genome_ids), len(vocab)), dtype=dtype)
#     return mat

# def load_labels(labels_path: Path, treat_intermediate_as_resistant: bool = True) -> pd.Series:
#     """Load phenotype labels from a CSV and return a binary series indexed by GenomeID.

#     Expects a CSV with at least `GenomeID` and `phenotype` columns. Optionally maps 'I' -> 'R'.

#     Parameters:
#     - labels_path: Path to CSV containing labels.
#     - treat_intermediate_as_resistant: if True, map 'I' to 'R' before filtering.

#     Returns:
#     - pandas Series indexed by GenomeID with values 1 for resistant and 0 for susceptible.
#     """
#     df = pd.read_csv(labels_path)
#     if 'GenomeID' not in df.columns or 'phenotype' not in df.columns:
#         raise ValueError('labels CSV must contain GenomeID and phenotype columns')
#     pheno = df.set_index('GenomeID')['phenotype'].str.upper()
#     if treat_intermediate_as_resistant:
#         pheno = pheno.replace({'I': 'R'})
#     pheno = pheno[pheno.isin(['R', 'S'])]
#     y = pheno.map({'R': 1, 'S': 0})
#     return y

## Feature Selection and Model Training

In [ ]:
# Sample 500 Resistant + 500 Susceptible and build sparse matrix 
seed = 42
from pathlib import Path
import random

labels_path = Path('../data/phenotype/ampicillin_phenotype.csv')
labels = load_labels(labels_path)
labels = labels.dropna()
labels = labels.astype(float)

res_ids = labels[labels == 1.0].index.astype(str).tolist()
sus_ids = labels[labels == 0.0].index.astype(str).tolist()

n_per_class = 500
if len(res_ids) < n_per_class or len(sus_ids) < n_per_class:
    raise ValueError(f'Not enough genomes to sample: have {len(res_ids)} R, {len(sus_ids)} S')

random.seed(seed)
sampled_res = random.sample(res_ids, n_per_class)
sampled_sus = random.sample(sus_ids, n_per_class)
sampled_ids = sampled_res + sampled_sus

out_ids_path = Path('../data/phenotype/ampicillin_1000_ids.txt')
out_ids_path.parent.mkdir(parents=True, exist_ok=True)
with out_ids_path.open('w') as fh:
    fh.write('\n'.join(sampled_ids))

print(f'Sampled {len(sampled_ids)} genomes (R={n_per_class}, S={n_per_class}), saved to {out_ids_path}')
